# Crawling Data Detik.com

Pada tahap ini dilakukan proses pengumpulan data berita secara otomatis (*web crawling*) dari portal **Detik.com**. Data yang diambil berupa judul dan isi berita dari dua kategori, yaitu **SPORT** dan **FINANCE**.

Beberapa *library* Python digunakan untuk mendukung proses pengambilan dan pengolahan data, yaitu:

- **Requests** digunakan untuk mengakses halaman web dan mengambil kode HTML.
- **BeautifulSoup** digunakan untuk membaca struktur HTML serta mengambil elemen yang diperlukan, seperti judul dan URL berita.
- **Pandas** digunakan untuk mengolah hasil *crawling* dan menyusunnya ke dalam bentuk tabel (*DataFrame*).
- **Trafilatura** digunakan untuk mengambil teks utama dari halaman berita secara otomatis agar isi yang diperoleh lebih bersih dan relevan.



In [1]:
!pip install requests beautifulsoup4 pandas trafilatura

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import trafilatura
import random

In [3]:




# Kategori berita
kategori_list = ['sport', 'finance']

# Jumlah data yang diambil dari setiap kategori
target_per_kategori = 100

# Menyimpan seluruh hasil crawling
data_berita = []

headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/131.0.0.0 Safari/537.36'
    )
}

# Halaman awal dibuat berbeda agar data tidak selalu sama
halaman_awal = {
    'sport': 6,
    'finance': 9
}

for kategori in kategori_list:

    print(f"\n=== Crawling {kategori.upper()} ===")

    halaman = halaman_awal[kategori]
    terkumpul = 0
    link_tersimpan = set()

    while terkumpul < target_per_kategori:

        url = f'https://{kategori}.detik.com/indeks?page={halaman}'

        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=15
            )

            if response.status_code != 200:
                print(f"Halaman {halaman} gagal diakses ({response.status_code})")
                halaman += 1
                continue

            soup = BeautifulSoup(response.text, 'html.parser')

            articles = soup.find_all('article')

            if not articles:
                articles = soup.select(
                    '.list-content article, .media__text'
                )

            if not articles:
                print(f"Tidak ada artikel pada halaman {halaman}.")
                halaman += 1
                continue

            # Urutan artikel dibuat berbeda
            random.shuffle(articles)

            for article in articles:

                if terkumpul >= target_per_kategori:
                    break

                title_tag = article.find(['h2', 'h3'])
                link_tag = article.find('a')

                if not title_tag or not link_tag:
                    continue

                tautan = link_tag.get('href')

                if not tautan:
                    continue

                # Menghindari URL yang sama
                if tautan in link_tersimpan:
                    continue

                judul = title_tag.get_text(strip=True)

                # Mengambil isi berita
                downloaded = trafilatura.fetch_url(tautan)

                if not downloaded:
                    continue

                isi_berita = trafilatura.extract(
                    downloaded,
                    include_comments=False,
                    include_tables=False
                )

                # Lewati artikel jika isi tidak berhasil diperoleh
                if not isi_berita:
                    continue

                # Simpan URL agar tidak terjadi duplikasi
                link_tersimpan.add(tautan)

                data_berita.append({
                    'Kategori': kategori.upper(),
                    'Judul Berita': judul,
                    'Isi Berita': isi_berita
                })

                terkumpul += 1

                print(
                    f"[{kategori.upper()}] "
                    f"{terkumpul}/{target_per_kategori} - {judul[:60]}"
                )

                # Jeda kecil antar request
                time.sleep(0.5)

            halaman += 1

        except Exception as e:
            print(f"Terjadi kesalahan pada halaman {halaman}: {e}")
            halaman += 1

        time.sleep(1.5)


# Membuat DataFrame
df_berita = pd.DataFrame(data_berita)

# Menghapus data duplikat berdasarkan judul
df_berita.drop_duplicates(
    subset=['Judul Berita'],
    inplace=True
)

# Mengacak urutan hasil akhir
df_berita = df_berita.sample(
    frac=1,
    random_state=190
).reset_index(drop=True)

# Menyimpan dataset
nama_file = 'data_berita_sport_finance_saya.csv'

df_berita.to_csv(
    nama_file,
    index=False
)

print("\n=== PROSES SELESAI ===")
print(f"Total data: {len(df_berita)} baris")
print(df_berita['Kategori'].value_counts())

display(df_berita.head())


=== Crawling SPORT ===
[SPORT] 1/100 - IHR 2026 Perluas Olahraga Pacuan Kuda Indonesia
[SPORT] 2/100 - Ambisi Morgan Holindo Jadi Juara Nasional Eshark Rok Cup 202
[SPORT] 3/100 - Kevin Sanjaya Antusias Lihat Semangat Talenta Bulutangkis In
[SPORT] 4/100 - Asian Games 2026: Banjir Landa Nagoya, Menpora Yakin Jepang 
[SPORT] 5/100 - Dari Hyrox ke 10K, Fanny Ghassani Pilih Pace Santai
[SPORT] 6/100 - BNI Dukung Skuad Bulutangkis Indonesia Menuju Asian Games 20
[SPORT] 7/100 - Rizki Juniansyah Siap Debut di Asian Games 2026, Naik Kelas 
[SPORT] 8/100 - CdM Todotua: Kami Monitoring Terus Kondisi di Nagoya
[SPORT] 9/100 - Atlet Muda Modern Pentathlon Punya Kesempatan Karier di Mili
[SPORT] 10/100 - Men's World Tennis Championship: Hari Baik untuk Wakil RI
[SPORT] 11/100 - Ketum KOI dan Menpora Mengukuhkan Tim Indonesia untuk Asian 
[SPORT] 12/100 - MotoGP San Marino 2026: Jaga Puncak Klasemen Bukan Prioritas
[SPORT] 13/100 - Pelatih Timnas Voli Putra RI Targetkan Medali di Asian Games
[SPO

,Kategori,Judul Berita,Isi Berita
0,SPORT,Ketum KOI Memastikan Atlet Terlayani dengan Ba...,"Ketua Umum Komite Olimpiade Indonesia (KOI), R..."
1,SPORT,Men's World Tennis Championship: Karan Singh J...,Selesai sudah Seri IV M-15 Men's World Tennis ...
2,FINANCE,Suahasil Ungkap Nasib Perombakan Pejabat Era P...,Menteri Keuangan Suahasil Nazara buka-bukaan n...
3,SPORT,Timnas Basket Jadi Gelombang Pertama Datang ke...,Timnas basket putra Indonesia jadi rombongan p...
4,SPORT,Ambisi Morgan Holindo Jadi Juara Nasional Esha...,"Pegokar muda Indonesia, Morgan Holindo, tak pu..."


Hasil pada tabel di atas menunjukkan data berita yang telah berhasil dikumpulkan dari halaman indeks. Data tersebut masih berupa data awal sehingga perlu melalui proses pembersihan dan pengolahan teks sebelum digunakan pada tahap berikutnya, yaitu *text preprocessing* dalam proses Web Mining.